[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jalonso1979/puremacro/blob/main/curso/notebooks/T03_E_local_llm_uncertainty.ipynb)

> **Google Colab note**: the next cell installs `puremacro` and, if needed, fetches only the `curso/notebooks` folder (helpers, frozen data and models) from the public repository [jalonso1979/puremacro](https://github.com/jalonso1979/puremacro). It does not ask for Google Drive access.

In [1]:
# Environment setup: Google Colab, local install or Juno (iPad)
import os
import sys
from pathlib import Path

def _safe_probe(p, is_dir=None):
    """Does the path exist? Never raises OSError/PermissionError in the Juno sandbox."""
    try:
        p = Path(p)
        if is_dir is True:
            return p.is_dir()
        if is_dir is False:
            return p.is_file()
        return p.exists()
    except (OSError, PermissionError, TypeError, ValueError):
        return False

if "google.colab" in sys.modules:
    !pip install -q "puremacro>=4.0.1,<5" openpyxl
    if not _safe_probe(Path.cwd() / "_nbstyle.py", is_dir=False):
        # Partial clone of the public repository: only curso/notebooks, no history
        _repo = Path("/content/puremacro")
        if not _safe_probe(_repo, is_dir=True):
            !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/jalonso1979/puremacro.git {_repo}
        !git -C {_repo} sparse-checkout set curso/notebooks 2>&1 | tail -n 2
        os.chdir(_repo / "curso/notebooks")

# Put the course helpers folder (_nbstyle, _datos) on sys.path, without leaving the sandbox
_candidates = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd() / "curso/notebooks"]
try:
    _candidates.insert(0, Path(__file__).resolve().parent)  # Juno defines __file__
except (NameError, OSError):
    pass
for _c in _candidates:
    if _safe_probe(_c / "_nbstyle.py", is_dir=False):
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

# Inline figures and the course style
%matplotlib inline
try:
    import _nbstyle
    _nbstyle.apply_style()
except ImportError:
    pass

# Large Language Models as Measurement Tools in Macroeconomics

Modern macroeconomic research frequently extracts quantitative indicators from unstructured narrative text: central bank policy statements, legislative statutes, beige books, and corporate disclosures.

Historically, text-as-data relied on rule-based dictionary lookups (e.g., Baker-Bloom-Davis EPU; Loughran & McDonald 2011). While computationally trivial and fully deterministic, dictionary methods suffer from fundamental econometric limitations:
1. **Negation and Contextual Reversal**: A statement such as *"The committee does not anticipate any near-term rise in inflation"* triggers positive counts in an inflation dictionary, inverting the economic sentiment.
2. **Polysemy and Syntactical Ambiguity**: The term *"rate"* refers to interest rates, unemployment rates, or foreign exchange rates depending entirely on surrounding syntax.
3. **Temporal Inflexibility**: Pre-compiled wordlists fail to adapt as policy rhetoric evolves (e.g., forward guidance terms such as *"liftoff"*, *"quantitative tightening"*, or *"flexible average inflation targeting"*).

Large Language Models (LLMs) address these limitations through dense self-attention mechanisms (Vaswani et al. 2017), projecting textual inputs into continuous semantic representations capable of parsing syntax, intent, and contextual qualification.

However, incorporating generative models into empirical macroeconomics introduces severe econometric and institutional dilemmas:
- **Data Privacy & Legal Confidentiality**: Uploading market-sensitive central bank deliberations, confidential bank supervision records, or proprietary corporate data to external cloud APIs violates legal charters and security regulations.
- **Scientific Replicability**: Commercial cloud APIs undergo continuous retraining, parameter updates, and alignment interventions without notice, destroying exact econometric replicability across research teams and time.
- **Stochasticity and Hallucination**: Without rigorous parameter constraints (such as setting sampling temperature to zero) and probability calibration, model outputs introduce measurement error that attenuates downstream regression coefficients.

This notebook analyzes the econometrics of **local, self-contained language model inference** (Apple MLX, llama.cpp, quantized GGUF architectures). We evaluate semantic information extraction, contrast dictionary matching with zero-shot LLM parsing on fiscal policy shocks, measure continuous narrative uncertainty via probability kernels, and establish best practices for replicable macro measurement.

In [2]:
import sys
from pathlib import Path

_cwd = Path.cwd()
sys.path.insert(0, str(_cwd if _safe_probe(_cwd / "_nbstyle.py", is_dir=False) else _cwd / "notebooks"))
import _nbstyle
_nbstyle.apply_style()

from puremacro.narrative.scoring import (
    get_default_backend,
    score_llm,
    score_keyword,
)
from puremacro.narrative.indices import (
    get_default_provider,
    llm_prob_kernel,
    MockProvider,
)

# Macroeconomic benchmark corpus: policy interventions, uncertainty dispatches, and neutral reporting
CORPUS = [
    (
        "2020-03-15",
        "The federal government enacted a $500 billion emergency fiscal stimulus and infrastructure package to support domestic liquidity.",
        "https://macro-corpus.example/doc_001",
    ),
    (
        "2020-04-01",
        "Central bank officials warned that the macroeconomic outlook is highly uncertain and could deteriorate abruptly if financial stress persists.",
        "https://macro-corpus.example/doc_002",
    ),
    (
        "2021-06-10",
        "The trade ministry reported that the bilateral trade surplus expanded moderately during the second quarter as port operations normalized.",
        "https://macro-corpus.example/doc_003",
    ),
    (
        "2022-09-22",
        "The committee announced a 75 basis point rate increase, noting that monetary tightening would continue until price stability is restored.",
        "https://macro-corpus.example/doc_004",
    ),
]

print(f"Corpus initialized with {len(CORPUS)} representative macroeconomic documents.")

Corpus initialized with 4 representative macroeconomic documents.


## 1. Local Offline Architecture: Guaranteeing Research Reproducibility and Data Security

In macroeconomic policy and central banking research, textual analytics must satisfy two operational constraints:
1. **On-Premise Isolated Deployment**: Computations must execute locally on the researcher's workstation or secure institutional cluster (using Apple MLX on Apple Silicon, llama.cpp, or local quantized models like Qwen-2.5-3B-Instruct or Llama-3.2-3B). Zero tokens leave the local memory environment, maintaining full compliance with central bank and national statistical confidentiality standards.
2. **Deterministic Offline Fallbacks**: In automated continuous integration (CI) environments or systems without GPU acceleration, `puremacro` provides deterministic mock providers (`MockBackend`, `MockProvider`). This decouples architectural validation and econometric unit testing from specific hardware requirements.

In [3]:
# Initialize local inference backend and narrative probability provider.
# puremacro auto-detects MLX (Apple Silicon) -> llama.cpp -> Ollama,
# falling back gracefully to deterministic offline providers in headless CI environments.
backend = get_default_backend("qwen2.5-3b-instruct")
provider = get_default_provider("qwen2.5-3b-instruct")

print(f"Active NLP Backend:       {backend.__class__.__name__}")
print(f"Active Narrative Provider: {provider.__class__.__name__}")

[puremacro] No local LLM engine available. Install one with `pip install puremacro[local-llm]` (MLX on Apple Silicon, or llama-cpp-python anywhere), or start Ollama (https://ollama.com) and run `ollama pull qwen2.5:3b`.
[puremacro] using MockBackend (zero events).
[puremacro] No local LLM engine available. Install one with `pip install puremacro[local-llm]` (MLX on Apple Silicon, or llama-cpp-python anywhere), or start Ollama (https://ollama.com) and run `ollama pull qwen2.5:3b`.
[puremacro] using MockProvider.
Active NLP Backend:       MockBackend
Active Narrative Provider: MockProvider


## 2. Information Extraction: Dictionaries vs Semantic Parsing in Fiscal Policy Shocks

A classic problem in empirical macroeconomics (Romer & Romer 2010; Cloyne 2013) is isolating exogenous fiscal actions from endogenous automatic stabilizers. Researchers must extract three key variables from legislative text:
1. **Date of Action**: When the policy was enacted or announced.
2. **Direction (Sign)**: Expansionary ($+1$) versus contractionary ($-1$) fiscal interventions.
3. **Magnitude and Units**: The fiscal impulse in billions of dollars or percent of GDP.

Below we contrast:
- **Dictionary & Regex Matching** (`score_keyword`): Applies predefined fiscal lexicons and regular expression number extractors.
- **Semantic LLM Extraction** (`score_llm`): Uses structured zero-shot prompting to parse dates, sign, targets (e.g., infrastructure vs transfers), and quantitative magnitudes directly into `NarrativeEvent` data structures.

Without a local language engine installed, `get_default_backend` falls back to `MockBackend`, which answers with an empty list: the output then shows zero events for the semantic method and only the dictionary extracts the $500 billion stimulus from the March 2020 document. The comparison between the two methods is only informative with a real model loaded.

In [4]:
# 1. Dictionary-based extraction (rule-based keyword matching)
events_kw = score_keyword(CORPUS, kind="fiscal")
print(f"Dictionary Method: Extracted {len(events_kw)} fiscal event(s)")
for ev in events_kw:
    print(f"  [{ev.date.date()}] Target: {ev.target}/{ev.subtarget} | Sign: {ev.sign:+d} | Magnitude: {ev.magnitude} {ev.magnitude_unit}")

# 2. Semantic LLM extraction (zero-shot structured parsing)
events_llm = score_llm(CORPUS, backend=backend, kind="fiscal")
print(f"Semantic LLM Method: Extracted {len(events_llm)} fiscal event(s)")
for ev in events_llm:
    print(f"  [{ev.date.date()}] Sign: {ev.sign:+d} | Magnitude: {ev.magnitude} {ev.magnitude_unit}")

# Econometric verification:
# Dictionary baseline reliably flags the explicit infrastructure investment shock in Document 0
assert len(events_kw) >= 1, "Dictionary method should identify explicit fiscal events"
assert events_kw[0].sign == 1, "Fiscal stimulus must be flagged as expansionary (sign=+1)"

Dictionary Method: Extracted 1 fiscal event(s)
  [2020-03-15] Target: investment/infra | Sign: +1 | Magnitude: 500.0 USD_bn
Semantic LLM Method: Extracted 0 fiscal event(s)


## 3. Narrative Uncertainty: Probability Kernels and Prompt Sensitivity

Beyond discrete event extraction, macroeconomists frequently require continuous text-based probability measures:
$$ P(\text{Uncertainty} \mid d_t) \in [0, 1] $$
representing the model's posterior probability that document $d_t$ conveys macroeconomic ambiguity.

`llm_prob_kernel()` calculates normalized logit probabilities over defined semantic categories. Without a local engine, `MockProvider` returns 1 when the text contains the word *uncertain* and 0 otherwise; that is why the output assigns probability 1 only to the April 2020 dispatch.

### Prompt Sensitivity and Temperature Calibration
In econometric applications, language models must be configured under strict statistical protocols:
1. **Greedy Decoding ($\tau = 0$)**: In generative modeling, non-zero temperature introduces sampling variance that renders classifications non-replicable across computational runs. Setting temperature to zero enforces deterministic argmax token selection.
2. **Category Boundary Specification**: Prompts must contain precise exclusion criteria (e.g., distinguishing fundamental economic policy uncertainty from ordinary business cycle reporting or routine central bank communications) to prevent semantic drift.

In [5]:
# Compute continuous semantic uncertainty scores across the corpus
prob_series = list(
    llm_prob_kernel(CORPUS, provider=provider, category="economic uncertainty")
)

print("Continuous Narrative Uncertainty Probabilities:")
for date, p in prob_series:
    print(f"  [{date.date()}] P(Economic Uncertainty | Document) = {p:.3f}")

# Verification assertions
probs_scored = [p for _, p in prob_series]

assert len(prob_series) == len(CORPUS), "Every document in the corpus must receive a score"
assert all(0.0 <= p <= 1.0 for p in probs_scored), "Probabilities must be strictly bounded in [0, 1]"

# Econometric hierarchy check:
# Document 1 (April 2020: 'outlook is highly uncertain') must score higher than Document 0 (March 2020: '$500B investment package')
if isinstance(provider, MockProvider):
    assert probs_scored[1] > probs_scored[0], (
        "Uncertainty warning (Doc 1) must receive higher uncertainty probability than investment stimulus (Doc 0)"
    )
    assert probs_scored[1] == 1.0 and probs_scored[0] == 0.0

Continuous Narrative Uncertainty Probabilities:
  [2020-03-15] P(Economic Uncertainty | Document) = 0.000
  [2020-04-01] P(Economic Uncertainty | Document) = 1.000
  [2021-06-10] P(Economic Uncertainty | Document) = 0.000
  [2022-09-22] P(Economic Uncertainty | Document) = 0.000


## Methodological Guidelines for LLM Measurement in Macroeconomics

To maintain scientific integrity when utilizing Large Language Models as econometric measurement instruments, researchers should adhere to four core principles:

1. **Deterministic Execution**: Always enforce greedy decoding ($\text{temperature} = 0$) and fix random seeds to guarantee exact empirical replicability.
2. **Local Hardware Deployment**: Run quantized open-weight models (MLX, GGUF via llama.cpp) locally on institutional infrastructure to prevent data leakage of confidential policy communications and eliminate dependence on third-party cloud APIs.
3. **Cross-Validation with Dictionaries and Human Annotations**: Validate zero-shot model outputs against established rule-based benchmarks (such as Baker-Bloom-Davis EPU) and blind expert human audits. Compute classification confusion matrices, precision, recall, and Cohen's kappa.
4. **Accounting for Measurement Error in Downstream Econometrics**: Treat NLP-generated variables as regressors measured with error ($x_t^* = x_t + \eta_t$). Use instrumental variables or errors-in-variables adjustments (e.g., Fuller 1987) to prevent attenuation bias when estimating macroeconomic dynamic response coefficients in VAR or local projection models.